In [ ]:
import pandas as pd

In [ ]:
df= pd.read_excel('Medical_Data_For_Fine_Tuning.xlsx')

In [ ]:
df.head()

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import pandas as pd
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"
dtype = torch.bfloat16

print(" Loading base model with Native FlashAttention-2 onto MI300X ")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    attn_implementation="flash_attention_2", 
    device_map="cuda:0"                       
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16, lora_alpha=16, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

df = pd.read_excel("Medical_Data_For_Fine_Tuning.xlsx")
dataset = Dataset.from_pandas(df)

def format_prompts(examples):
    texts = []
    for inp, out in zip(examples["Input_Narrative"], examples["Structured_Output_JSON"]):
        messages = [
            {"role": "system", "content": "Extract PV data into a single-line flattened JSON string."},
            {"role": "user", "content": str(inp)},
            {"role": "assistant", "content": str(out)}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

training_args = SFTConfig(
    output_dir="hf_outputs",
    per_device_train_batch_size=16,  
    gradient_accumulation_steps=1,    
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    bf16=True,                       
    logging_steps=1,
    dataloader_num_workers=4,         
    report_to="none",
    dataset_text_field="text",        
    max_length=2048                   
)

print(" Running training loop at maximum hardware capability ")
trainer = SFTTrainer(
    model=model, 
    train_dataset=dataset, 
    args=training_args                
)
trainer.train()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-3B-Instruct"
adapter_dir = "hf_outputs/checkpoint-60" 

print(" Loading base Qwen model onto clean MI300X memory ")
dtype = torch.bfloat16
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=dtype,
    attn_implementation="flash_attention_2", 
    device_map="cuda:0"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

print(" Merging fine-tuned LoRA adapters ")
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval() 

test_narrative = (
    "A 62-year-old female patient with a history of chronic plaque psoriasis was started on "
    "Secukinumab (Cosentyx) 300mg weekly. Following the fourth loading dose, she experienced a sudden "
    "onset of severe watery diarrhea, intense abdominal cramping, head spinning and a low-grade fever (38.1°C)."
)

messages = [
    {"role": "system", "content": "Extract PV data into a single-line flattened JSON string."},
    {"role": "user", "content": test_narrative}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda:0")

print("\n Executing generation pass ")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=256, 
        do_sample=False,              
        eos_token_id=tokenizer.eos_token_id
    )

generated_ids = outputs[0][inputs.input_ids.shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("\n🟢 FINE-TUNED MODEL OUTPUT:")
print("=" * 80)
print(response)
print("=" * 80)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Configuration
base_model_id = "Qwen/Qwen2.5-3B-Instruct"
adapter_dir = "hf_outputs/checkpoint-60" 

hf_repository_id = "Bokkisam-Rohit24/Qwen2.5-3B-Medical-PV-Extract"
hf_write_token = "<Our-HF-Token>"

print("⏳ Loading base model and tokenizer onto MI300X...")
dtype = torch.bfloat16
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=dtype,
    device_map="cuda:0"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("⚡ Attaching your fine-tuned LoRA adapters...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print(" Merging adapter layers into a single standalone model ")
merged_model = model.merge_and_unload()

print(f" Uploading fully merged model to Hugging Face Hub under: {hf_repository_id}")

# Push the weights
merged_model.push_to_hub(
    repo_id=hf_repository_id, 
    token=hf_write_token,
    private=False
)
ss
tokenizer.push_to_hub(
    repo_id=hf_repository_id, 
    token=hf_write_token
)

print("\n Success Your model is live on Hugging Face and ready to pull down to your laptop.")